In [1]:
import pandas as pd
import os
from langchain.llms import HuggingFacePipeline
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma
from langchain.document_loaders import PyMuPDFLoader
from langchain.chains import RetrievalQA
from tqdm import tqdm
from langchain.prompts import PromptTemplate
import time

In [2]:
embedding = SentenceTransformerEmbeddings(model_name="chuxin-llm/Chuxin-Embedding")

C:\Users\Jacky\AppData\Local\Temp\ipykernel_11052\590070299.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = SentenceTransformerEmbeddings(model_name="chuxin-llm/Chuxin-Embedding")
C:\Users\Jacky\AppData\Roaming\Python\Python39\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [6]:
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
pdf_directory = "c:/Users/Jacky/Desktop/industrial_project/dataset/chin_datasets"
pdf_names = pdf_names = [pdf_name for pdf_name in os.listdir(pdf_directory) if pdf_name.endswith('.pdf')]
split_documents = []
for pdf_name in tqdm(pdf_names, desc="Loading PDFs"):
    pdf_path = os.path.join(pdf_directory, pdf_name)
    loader = PyMuPDFLoader(pdf_path)
    loaded_documents = loader.load()
    for document in loaded_documents:
        split_texts = document.page_content.split("\n\n")
        for texts in split_texts:
            split_documents.append(Document(page_content=texts.strip(), metadata=document.metadata))

Loading PDFs: 100%|██████████| 7/7 [00:01<00:00,  3.85it/s]


In [7]:
vector_store = Chroma.from_documents(split_documents, embedding)
retriever=vector_store.as_retriever(search_kwargs={"k": 3})

In [21]:
question_df = pd.read_excel("c:/Users/Jacky/Desktop/industrial_project/dataset/test.xlsx")

In [22]:
question_df['pdf'].unique()

array(['GS-1', 'Guideline_on_AML-CFT_(for_AIs)_chi_May 2023', 'SPM-AML-1',
       'CG-1-Ch', 'SA-1-Ch', 'SA-2-Ch', 'TM-E-1', 'TM-G-1', 'TM-G-2'],
      dtype=object)

In [34]:
docs[0].metadata['page']

17

In [23]:
chin_question = question_df[question_df['pdf']!='GS-1']
chin_question = chin_question[chin_question['pdf']!='TM-E-1']

In [24]:
len(chin_question)

70

In [26]:
relevant_content = []
for query in tqdm(chin_question['query']):
    docs = retriever.get_relevant_documents(query)
    relevant_content.append([docs[i].page_content for i in range(len(docs))])

100%|██████████| 70/70 [00:24<00:00,  2.90it/s]


In [28]:
chin_question['relevant_content'] = relevant_content

In [30]:
chin_question.to_excel('chin_retrieve_single_pdf.xlsx')

In [ ]:
question = pd.read_excel('/kaggle/input/test-data/test.xlsx')

In [7]:
def rag_once(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = inputs.to('cuda')
    input_token = len(tokenizer(prompt)["input_ids"])
    outputs = model.generate(**inputs,max_length=input_token+200)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split('Answer:\n')[1]
    return answer

In [8]:
def prompt_rag(context,question):
    return f"""
    使用所給的上下文來回答問題，並且可以使用已有的知識。
    使用不超過三句來回答，並且用繁體中文。
    上下文: {context}
    Question: {question}
    Answer:
    """

In [9]:
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
judge_prompt = PromptTemplate(
    template="""You are a grader assessing relevance of a retrieved document to a user question. \n 
    Here is the retrieved document: \n\n {document} \n\n
    Here is the user question: {question} \n
    If the document contains keywords related to the user question, grade it as relevant. \n
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question. \n
    Provide the binary score as a JSON with a single key 'score' and no premable or explanation.""",
    input_variables=["question", "document"],
)
rag_prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. \n
    If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
    Question: {question} 
    Context: {context} 
    Answer:""",
    input_variables=["question", "context"],
)
hac_prompt = PromptTemplate(
    template="""You are a grader assessing whether an answer is grounded in / supported by a set of facts. \n 
    Here are the facts:
    \n ------- \n
    {documents} 
    \n ------- \n
    Here is the answer: {generation}
    Give a binary score 'yes' or 'no' score to indicate whether the answer is grounded in / supported by a set of facts. \n
    Provide the binary score as a JSON with a single key 'score' and no preamble or explanation.""",
    input_variables=["generation", "documents"],
)
rewrite_prompt = PromptTemplate(
    template="""You a question re-writer that converts an input question to a better version that is optimized \n 
     for vectorstore retrieval. Look at the initial and formulate an improved question. \n
     Here is the initial question: \n\n {question}.,
    Provide a improved question as a JSON with a single key 'question' and no premable or explanation.""",
    input_variables=["question"],
)
rel_prompt = PromptTemplate(
    template="""You are a grader assessing whether an answer is useful to resolve a question. \n 
    Here is the answer:
    \n ------- \n
    {generation} 
    \n ------- \n
    Here is the question: {question}
    Give a binary score 'yes' or 'no' to indicate whether the answer is useful to resolve a question. \n
    Provide the binary score as a JSON with a single key 'score' and no preamble or explanation.""",
    input_variables=["generation", "question"],
)

In [10]:
model_choice = 'gemma'
while True:
    user_question = input("Please enter your question (or press Enter to quit): ")
    if user_question == "":
        print("No question entered. Exiting.")
        break
    else:
        context = retriever.get_relevant_documents(user_question)
        retrieved_content = [item.page_content for item in context]
        prompt = prompt_rag(retrieved_content, user_question)
        if model_choice == "llama":
            response = rag_once_llama(prompt)
        else: 
            response = rag_once(prompt)
    print(f"Model Response: {response}")

Please enter your question (or press Enter to quit):  abc


<ipython-input-10-4f0f2becccc7>:8: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  context = retriever.get_relevant_documents(user_question)


Model Response:     抱歉，我無法理解您的問題。





Please enter your question (or press Enter to quit):  


No question entered. Exiting.


In [21]:
doc_txt = docs[0].page_content
prompt = judge_prompt.format(question = query, document = docs)
inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
input_token = len(tokenizer(prompt)["input_ids"])
outputs = model.generate(**inputs,max_length=input_token+50)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
result = JsonOutputParser().parse(response)

In [11]:
from langchain_core.output_parsers.json import JsonOutputParser
from langchain_core.output_parsers.json import OutputParserException
max_iter = 3
def is_valid_json(response):
    try:
        JsonOutputParser().parse(response)
        return True 
    except OutputParserException:
        return False 
        
def judge_document(query, docs):
    relevant_documents = []
    for i in range(len(docs)):
        doc_txt = docs[i].page_content
        prompt = judge_prompt.format(question = query, document = doc_txt)
        for attempt in range(max_iter):
            inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
            input_token = len(tokenizer(prompt)["input_ids"])
            outputs = model.generate(**inputs,max_length=input_token+50)
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            if is_valid_json(response):
                break
        parsed_response = JsonOutputParser().parse(response)
        if 'yes' in parsed_response['score']:
            print(f"檢索到第{docs[i].metadata['page']+1}頁, 此頁與問題相關")
            relevant_documents.append(doc_txt)
        else:
            print(f"檢索到第{docs[i].metadata['page']+1}頁, 此頁與問題無關")
    return relevant_documents
    
def generate_response(query, relevant_documents):
    prompt = rag_prompt.format(question = query, context = relevant_documents)
    inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
    input_token = len(tokenizer(prompt)["input_ids"])
    outputs = model.generate(**inputs,max_length=input_token+200)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = StrOutputParser().parse(result)
    return response
    
def judge_hac(response, relevant_documents):
    prompt = hac_prompt.format(generation = response, documents = relevant_documents)
    for attempt in range(max_iter):
        inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
        input_token = len(tokenizer(prompt)["input_ids"])
        outputs = model.generate(**inputs,max_length=input_token+50)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if is_valid_json(response):
            break
    parsed_response = JsonOutputParser().parse(response)
    return parsed_response
    
def judge_rel(response, query):
    prompt = rel_prompt.format(generation = response, question = query)
    for attempt in range(max_iter):
        inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
        input_token = len(tokenizer(prompt)["input_ids"])
        outputs = model.generate(**inputs,max_length=input_token+50)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if is_valid_json(response):
            break
    parsed_response = JsonOutputParser().parse(response)
    is_rel = parsed_response['score']
    return is_rel
def rewrite_question(query):
    prompt = rewrite_prompt.format(question = query)
    for attempt in range(max_iter):
        inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
        input_token = len(tokenizer(prompt)["input_ids"])
        outputs = model.generate(**inputs,max_length=input_token+50)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if is_valid_json(response):
            break
    parsed_response = JsonOutputParser().parse(response)
    query = parsed_response['question']
    return query

In [12]:
query = '''認可機構應如何進行營運層面的風險評估?'''
while True:
    print('Retrieving files')
    docs = retriever.get_relevant_documents(query)
    relevant_documents = judge_document(query, docs)
    if relevant_documents == []:
        print('No relevant files, we are rewriting the question now')
        print('Rewrite the problem as:')
        query = rewrite_question(query)
        print(query)
    else:
        print('There are relevant documents')
        print('Generating answers')
        response = generate_response(query, relevant_documents)
        print('Answer:', response)
        is_hac = judge_hac(response, relevant_documents)
        if 'yes' in is_hac:
            for i in range(3):
                if 'yes' in hac_result:
                    print('This answer is an hallucination, we need to generate a new answer')
                    continue
                else:
                    print('This answer is not an hallucination, we now determine if the answer helps solve the problem ')
                    break
        else:
            print('This answer is not an hallucination, we now determine if the answer helps solve the problem ')
            is_rel = judge_rel(response, query)
            if 'yes' in is_rel:
                print('This response helps resolve the query')
                break
            else:
                print('This response did not help resolve the query, we are now rewriting the question')
                query = rewrite_question(query)
                print('Rewrite the problem as:')
                print(query)

Retrieving files
檢索到第11頁, 此頁與問題相關
There are relevant documents
Generating answers
Answer: You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. 

    If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
    Question: 認可機構應如何進行營運層面的風險評估? 
    Context: ['9\n \n2.8 \n就第2.2 及2.7 段而言，如認可機構為某金融集團的成\n員，並已進行集團層面或地區層面的洗錢及恐怖分子資\n金籌集風險評估，則該認可機構可參考或依賴有關評\n估，惟有關評估須充分反映該認可機構在本地層面所面\n對的洗錢及恐怖分子資金籌集風險。 \n \n \n2.9 \n為確保機構層面的洗錢及恐怖分子資金籌集風險評估能\n反映現況，認可機構應每兩年，以及在發生對其業務及\n所面對風險有重大影響的觸發事件時進行有關評估。 \n \n新產品、新經營方法及使用新科技 \n \n2.10 \n認可機構應識別及評估因以下情況而可能產生的洗錢及\n恐怖分子資金籌集風險： \n \n(a) 開發新產品及新經營方法，包括新的交付機制；及 \n(b) 就新產品及既有產品使用新的或開發中的科技。 \n \n \n2.11 \n認可機構在推出新產品、新經營方法或使用新的或發展\n中的科技前，應事先作出風險評估，並應採取適當措施\n管理及緩減所識別的風險。 \n \n客戶風險評估 \n \n2.12 \n認可機構應評估擬建立的業務關係涉及的洗錢及恐怖分\n子資金籌集風險，這類評估一般稱為客戶風險評估。在\n盡職審查程序初期所作的評估，將決定需採取何種程度\n的盡職審查措施3。即是說，如有關業務關係涉及的洗錢\n及恐怖分子資金籌集風險較高，所索取資料的數量及類\n別便會較多，核實資料的程度亦會提升。如有關業務關

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.76 GiB. GPU 0 has a total capacity of 14.74 GiB of which 1.15 GiB is free. Process 4804 has 13.59 GiB memory in use. Of the allocated memory 12.77 GiB is allocated by PyTorch, and 706.60 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [10]:
torch.cuda.empty_cache()

In [ ]:
def ircot_prompt(question: str, context: str, retrieved_info: str) -> str:
    input_text = f"""
    Analyze the question and retrieved information step-by-step. Use the following format:

    Question: {question}
    Context: {context}
    Retrieved Information: {retrieved_info}

    Reasoning:
    1. Start by addressing the question directly.
    2. Reference key details from the retrieved information.
    3. Connect the dots between the question and retrieved facts.
    4. If confident, conclude with "The answer is: [answer]".
    """
    return input_text

In [ ]:
def generate_reasoning(retrieved_context, context, question):
    prompt = ircot_prompt(retrieved_context, context, question)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = inputs.to('cuda')
    input_token = len(tokenizer(prompt)["input_ids"])
    outputs = model.generate(**inputs,max_length=input_token+200)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [ ]:
def ircot(question):
    max_iter = 3
    context = ''
    history = {
        "queries": [],
        "retrieved_docs": [],
        "reasoning": [],
    }

    for iteration in range(1, max_iter + 1):
        print(f"\n=== Iteration {iteration} ===")

        # Step 1: Generate search query
        if iteration == 1:
            query = question  # Start with the original question
        else:
            query = f"{question} based on {history['reasoning'][-1]}"  # Refine query based on previous reasoning
        history["queries"].append(query)
        print(f"Search Query: {query}")

        # Step 2: Retrieve relevant documents
        retrieved_docs = retriever.get_relevant_documents(query)
        history["retrieved_docs"].append(retrieved_docs)
        retrieved_info = "\n".join([doc.page_content for doc in retrieved_docs])  # Combine retrieved documents

        # Step 3: Generate reasoning using the Hugging Face model
        reasoning = generate_reasoning(question, context, retrieved_info)
        history["reasoning"].append(reasoning)
        print(f"Reasoning: {reasoning}")

        # Update context for the next iteration
        context += f"\nIteration {iteration} Reasoning: {reasoning}"

        # Step 4: Check if further iterations are needed
        if "answer is" in reasoning.lower() or iteration >= max_iter:
            print("\nStopping iterations. Finalizing answer.")
            break

    # Final answer
    final_answer = history["reasoning"][-1]
    print(f"\n=== Final Answer ===\n{final_answer}")
    return history

In [ ]:
query = question['query'][0]
torch.cuda.empty_cache()
answer = ircot(query)

In [ ]:
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
prompt = PromptTemplate(
    template="""You are a grader assessing relevance of a retrieved document to a user question. \n 
    Here is the retrieved document: \n\n {document} \n\n
    Here is the user question: {question} \n
    If the document contains keywords related to the user question, grade it as relevant. \n
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question. \n
    Provide the binary score as a JSON with a single key 'score' and no premable or explanation.""",
    input_variables=["question", "document"],
)
rag_prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
    Question: {question} 
    Context: {context} 
    Answer:""",
    input_variables=["question", "document"],
)
hac_prompt = PromptTemplate(
    template="""You are a grader assessing relevance of a retrieved document to a user question. \n 
    Here is the retrieved document: \n\n {document} \n\n
    Here is the user question: {question} \n
    If the document contains keywords related to the user question, grade it as relevant. \n
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question. \n
    Provide the binary score as a JSON with a single key 'score' and no premable or explanation.""",
    input_variables=["question", "document"],
)
# Function to generate output using the model
def generate_judge(question, document):
    input_text = prompt.format(question=question, document=document)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    # Generate output
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=800)  # Adjust max_length as needed

    # Decode the output and parse JSON
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return JsonOutputParser().parse(output_text)
def generate_output(question, document):
    input_text = prompt.format(question=question, document=document)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    # Generate output
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=4096)  # Adjust max_length as needed

    # Decode the output and parse JSON
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return JsonOutputParser().parse(output_text)


In [ ]:
query = question['query'][0]
docs = retriever.get_relevant_documents(query)
results = []
for i in range(3):
    doc_txt = docs[i].page_content
    result = generate_judge(query, doc_txt)
    if 'yes' in result['score']:
        results.append(doc_txt)

In [ ]:
result['score']


In [ ]:
def clean_text(text):
    cleaned = re.sub(r'\s+', ' ', text)
    return cleaned.strip()
def prompt_rag(context,question):
    return f"""
    使用所給的上下文來回答問題。
    使用不超過三句來回答，並且用繁體中文。
    上下文: {context}
    Question: {question}
    Answer:
    """

In [ ]:
maxi_iter = 3
question = "TCFD在風險管理上應有甚麼建議?"
retrieved_context = []
reason = question
for i in range(maxi_iter):
    retrieved_docs = retriever.get_relevant_documents(reason)
    context = retrieved_docs.page_content
    retrieved_context.append(clean_text(context))
    prompt = prompt_rag(retrieved_context, question)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = inputs.to('cuda')
    input_token = len(tokenizer(prompt)["input_ids"])
    outputs = model.generate(**inputs,max_length=input_token+100)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split('Answer:\n')[1]
    reason.append(answer)

In [ ]:
def get_chatbot_response(user_message):
    # Retrieve relevant documents
    retrieved_docs = retriever.get_relevant_documents(user_message)
    contexts = []
    for context in retrieved_docs:
        contexts.append(context.page_content)
    
    # Prepare the input for the language model
    prompt = prompt_rag(contexts,user_message)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = inputs.to('cuda')
    input_token = len(tokenizer(prompt)["input_ids"])
    outputs = model.generate(**inputs,max_length=input_token+100)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split('Answer:\n')[1]
    
    return answer

In [ ]:
context = retriever.get_relevant_documents(question['query'][0])

In [ ]:
context_retrieved = []
for query in question['query']:
    retrieved_context = retriever.get_relevant_documents(query)
    contexts = []
    for context in retrieved_context:
        contexts.append(context.page_content)
    context_retrieved.append(contexts)

In [ ]:
def prompt_ircot(context,question):
    return f"""
    Construct a global reasoning chain for this complex [Question] : "{Question}" You should generate a query to the search engine based on what you already know at each step of the reasoning chain, starting with [Query].
If you know the answer for [Query], generate it starting with [Answer].
You can try to generate the final answer for the [Question] by referring to the [Query]-[Answer] pairs, starting with [Final Content].
If you don't know the answer, generate a query to search engine based on what you already know and do not know, starting with [Unsolved Query].
For example:
[Question]: "Where do greyhound buses that are in the birthplace of Spirit If...'s performer leave from? "
[Query 1]: Who is the performer of Spirit If... ?
If you don't know the answer:
[Unsolved Query]: Who is the performer of Spirit If... ?
If you know the answer:
[Answer 1]: The performer of Spirit If... is Kevin Drew.
[Query 2]: Where was Kevin Drew born?
If you don't know the answer:
[Unsolved Query]: Where was Kevin Drew born?
If you know the answer:
[Answer 2]: Toronto.
[Query 3]: Where do greyhound buses in Toronto leave from?
If you don't know the answer:
[Unsolved Query]: Where do greyhound buses in Toronto leave from?
If you know the answer:
[Answer 3]: Toronto Coach Terminal.
[Final Content]: The performer of Spirit If... is Kevin Drew [1]. Kevin Drew was born in Toronto [2]. Greyhound buses in Toronto leave from Toronto Coach Terminal [3]. So the final answer is Toronto Coach Terminal.

[Question]:"Which magazine was started first Arthur’s Magazine or First for Women?"
[Query 1]: When was Arthur’s Magazine started?
[Answer 1]: 1844.
[Query 2]: When was First for Women started?
[Answer 2]: 1989
[Final Content]: Arthur’s Magazine started in 1844 [1]. First for Women started in 1989 [2]. So Arthur’s Magazine was started first. So the answer is Arthur’s Magazi
[Question]: "{Question}"
    """

In [ ]:
def prompt_rag(context,question):
    return f"""
    使用所給的上下文來回答問題。
    使用不超過三句來回答，並且用繁體中文。
    上下文: {context}
    Question: {question}
    Answer:
    """

In [ ]:
input_tokens = []
output_tokens = []
answers = []
prompts = []
times = []
for _, row in tqdm(question[['query']].iterrows()):
    import time
    start_time = time.time()
    query = row['query']
    answer = query
    every_prompts = []
    retrieved_context = []
    every_input_tokens = []
    every_output_tokens = []
    for i in range(3):
        context = retriever.get_relevant_documents(answer)
        text = clean_text(item.page_content)
        retrieved_context.append(text)
        prompt = prompt_rag(retrieved_context,query)
        every_prompts.append(prompt)
        inputs = tokenizer(prompt, return_tensors="pt")
        inputs = inputs.to('cuda')
        input_token = len(tokenizer(prompt)["input_ids"])
        every_input_tokens.append(input_token)
        outputs = model.generate(**inputs,max_length=input_token+100)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer = response.split('Answer:\n')[1]
        output_token = len(tokenizer(answer)["input_ids"])
        every_output_tokens.append(output_token)
        time_use = time.time() - start_time
    prompts.append(every_prompts)
    input_tokens.append(every_input_tokens)
    output_tokens.append(every_output_tokens)
    answers.append(answer)
    times.append(time_use)
    torch.cuda.empty_cache()

In [ ]:
def prompt_rag(context,question):
    return f"""
    使用所給的上下文來回答問題，並且可以使用已有的知識。
    使用不超過三句來回答，並且用繁體中文。
    上下文: {context}
    Question: {question}
    Answer:
    """
def ask_question_with_rag_allow_exist(promp):
    import time
    start_time = time.time()
    response = pipe(promp)
    answer = response[0]['generated_text'].split('Answer:')[1].replace('\n', ' ').strip()
    input_token = len(pipe.tokenizer(promp)["input_ids"])
    output_token = len(pipe.tokenizer(answer)["input_ids"])
    time = time.time() - start_time
    return input_token, output_token, answer, time

In [ ]:
def prompt_no_rag(question):
    return f"""
    使用不超過三句來回答問題，並且用繁體中文。
    Question: {question}
    Answer:
    """

In [ ]:
context['input_token'] = input_tokens
context['output_token'] = output_tokens
context['first_answer'] = answers
context['first_time'] = times

In [ ]:
context.to_excel('Qwen2.5-1.5B-instruct-0141.xlsx', index=False)

In [ ]:
input_tokens = []
output_tokens = []
answers = []
times = []
for index, row in tqdm(context[['query', 'retrived_text']].iterrows()):
    q = row['query']
    r = row['retrived_text']
    prompt = prompt_no_rag(q)
    input_token, output_token, answer, time = ask_question_with_rag_allow_exist(prompt)
    input_tokens.append(input_token)
    output_tokens.append(output_token)
    answers.append(answer)
    times.append(time)

In [ ]:
context['input_token'] = input_tokens
context['output_token'] = output_tokens
context['first_answer'] = answers
context['first_time'] = times

In [ ]:
model_name = "Qwen/Qwen2.5-3B-Instruct"
llm_pipeline  = pipeline("text-generation", model=model_name,max_length = 8192,max_new_tokens = 200, device=0, token=os.getenv('HF_TOKEN'))

In [ ]:
model_name = "google/gemma-2b-it"
llm_pipeline  = pipeline("text-generation", model=model_name,max_length = 8192,max_new_tokens = 200, device=0, token=os.getenv('HF_TOKEN'))

In [ ]:
test = pd.read_excel('/kaggle/input/llama-text/llama_3.2_instruct_0554.xlsx')


In [ ]:
def ask_question_rag_qwen(prompt):
    start_time = time.time()
    response = llm_pipeline(prompt+'Answer: ')
    answer = response[0]['generated_text'].split('Answer: ')[1].strip()
    input_token = len(llm_pipeline.tokenizer(prompt)["input_ids"])
    output_token = len(llm_pipeline.tokenizer(answer)["input_ids"])
    num_tokens = len(input_tokens)
    t = time.time() - start_time
    return input_token, output_token, answer, t

In [ ]:
input_tokens = []
output_tokens = []
first_answers = []
first_times = []
for prompt in tqdm(test['first_prompt']):
    input_token, output_token, first_answer, first_time = ask_question_rag_qwen(prompt)
    input_tokens.append(input_token)
    output_tokens.append(output_token)
    first_answers.append(first_answer)
    first_times.append(first_time)

In [ ]:
test['input_token'] = input_tokens
test['output_token'] = output_tokens
test['first_answer'] = first_answers
test['first_time'] = first_times

In [ ]:
test.to_excel('gemma_2_2b_it_0157.xlsx')

In [ ]:
def ask_question_qwen_no_rag(prompt):
    start_time = time.time()
    response = llm_pipeline(f"使用用繁體中文來回答。\n用這格式回答: Qustion:{prompt}\nAnswer: ")
    answer = response[0]['generated_text'].split('Answer:')[1].strip()
    input_token = len(llm_pipeline.tokenizer(prompt)["input_ids"])
    output_token = len(llm_pipeline.tokenizer(answer)["input_ids"])
    num_tokens = len(input_tokens)
    t = time.time() - start_time
    return input_token, output_token, answer, t

In [ ]:
input_tokens = []
output_tokens = []
first_answers = []
first_times = []
for query in tqdm(test['query']):
    input_token, output_token, first_answer, first_time = ask_question_qwen_no_rag(query)
    input_tokens.append(input_token)
    output_tokens.append(output_token)
    first_answers.append(first_answer)
    first_times.append(first_time)

In [ ]:
test['input_token'] = input_tokens
test['output_token'] = output_tokens
test['first_answer'] = first_answers
test['first_time'] = first_times

In [ ]:
test.to_excel('gemma_2_2b_it_baseline_0139.xlsx')

In [ ]:
file_path = '/kaggle/input/test-data/test.xlsx'
test = pd.read_excel(file_path)

In [ ]:
pdf_directory = "/kaggle/input/project"
pdf_names = os.listdir(pdf_directory)
documents = []
total_token_count = 0
for pdf_name in tqdm(pdf_names, desc="Loading PDFs"):
    pdf_path = os.path.join(pdf_directory, pdf_name)
    loader = PyMuPDFLoader(pdf_path)
    loaded_documents = loader.load()
    for document in loaded_documents:
        text = document.page_content
        documents.append(document)
        total_token_count+=len(llm_pipeline.tokenizer(text)["input_ids"])
print(f"Token count: {total_token_count}")

In [ ]:
vector_store = Chroma.from_documents(documents, embedding_model)

In [ ]:
retriever=vector_store.as_retriever()
prompt_template ="""
    使用所給的上下文來回答問題。
    使用不超過三句來回答，並且用繁體中文。
    Context: {context}
    Question: {question}
    Answer:
"""
prompt = PromptTemplate.from_template(prompt_template)
qa = RetrievalQA.from_chain_type(
    retriever=retriever,
    llm=HuggingFacePipeline(pipeline=llm_pipeline),
    chain_type="stuff",
    chain_type_kwargs = {"prompt": prompt}
)

In [ ]:
def ask_question_with_rag(question, max_retries=10, retry = False):
    start_time = time.time()
    response = qa({"query": question})
    first_prompt = response['result'].split('Answer:')[0].strip()
    first_answer = response['result'].split('Answer:')[1].strip()
    input_token = len(llm_pipeline.tokenizer(first_prompt)["input_ids"])
    output_token = len(llm_pipeline.tokenizer(first_answer)["input_ids"])
    first_time = time.time() - start_time
    if retry == False:
        return input_token, output_token, first_prompt, first_answer, first_time
    else:
        if not any(unwanted in first_answer for unwanted in ["不知", "I don't know", "不清楚", "None","不確定","不是在這個文本中提到"]):
            return first_tokens, first_token, first_prompt, first_prompt, first_answer, first_answer, 1, first_time, first_time
        for attempt in range(2, max_retries + 2):
            response = qa({"query": question})
            final_prompt = response['result'].split('Answer:')[0].strip()
            final_answer = response['result'].split('Answer:')[1].strip()
            if not any(unwanted in final_answer for unwanted in ["不知道", "I don't know", "不清楚", "None",'不是在這個文本中提到']):
                final_time = time.time() - start_time
                input_token = len(llm_pipeline.tokenizer(first_prompt)["input_ids"])
                output_token = len(llm_pipeline.tokenizer(first_answer)["input_ids"])
                return first_token, final_token, first_prompt, final_prompt, first_answer, final_answer, attempt, first_time, final_time

In [ ]:
def ask_question_not_rag(question):
    start_time = time.time()
    response = llm_pipeline('使用用繁體中文來回答。\n用這格式回答: Qustion:{question}\nAnswer:{answer}\nQustion:'+ question)
    answer = response[0]['generated_text'].split('Answer:')[2].strip()
    input_token = len(llm_pipeline.tokenizer(question)["input_ids"])
    output_token = len(llm_pipeline.tokenizer(answer)["input_ids"])
    num_tokens = len(input_tokens)
    t = time.time() - start_time
    return input_token, output_token, answer, t

In [ ]:
input_tokens = []
output_tokens = []
first_prompts = []
first_answers = []
first_times = []
for query in tqdm(test['query']):
    input_token, output_token, first_prompt, first_answer, first_time = ask_question_with_rag(query, retry = False)
    input_tokens.append(input_token)
    output_tokens.append(output_token)
    first_prompts.append(first_prompt)
    first_answers.append(first_answer)
    first_times.append(first_time)

In [ ]:
first_tokens = []
final_tokens = []
first_prompts = []
final_prompts = []
first_answers = []
final_answers = []
attempts = []
first_times = []
final_times = []
for query in tqdm(test['query']):
    first_token, final_token, first_prompt, final_prompt, first_answer, final_answer, attempt, first_time, final_time = \
    ask_question_with_rag(query, retry = True)
    first_tokens.append(first_token)
    final_tokens.append(final_token)
    first_prompts.append(first_prompt)
    final_prompts.append(final_prompt)
    first_answers.append(first_answer)
    final_answers.append(final_answer)
    attempts.append(attempt)
    first_times.append(first_time)
    final_times.append(final_time)

In [ ]:
test['input_token'] = input_tokens
test['output_token'] = output_tokens
test['first_prompt'] = first_prompts
test['first_answer'] = first_answers
test['first_time'] = first_times

In [ ]:
test['attempts'] = attempts
test['first_token'] = first_tokens
test['first_prompt'] = first_prompts
test['first_answer'] = first_answers
test['first_time'] = first_times
test['final_token'] = final_tokens
test['final_prompt'] = final_prompts
test['final_answer'] = final_answers
test['final_time'] = final_times

In [ ]:
import time
from tqdm import tqdm
answer = []
input_tokens = []
output_tokens = []
times = []
for query in tqdm(test['query']):
    input_token, output_token, response, t = ask_question_not_rag(query)
    times.append(t)
    answer.append(response)
    input_tokens.append(input_token)
    output_tokens.append(output_token)

In [ ]:
answer

In [ ]:
test['response'] = answer
test['input_token'] = input_tokens
test['output_token'] = output_tokens
test['time'] = times

In [ ]:
test.to_excel('llama_3.2_instruct_baseline_0440.xlsx', index=False)

In [ ]:
answer

In [ ]:
from IPython.core.display import display, HTML

html_code = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>RAG Chatbot</title>
    <style>
        body { font-family: Arial, sans-serif; }
        #chatbox { border: 1px solid #ccc; padding: 10px; height: 400px; overflow-y: scroll; }
        #userInput { width: 80%; }
    </style>
</head>
<body>
    <h1>RAG Chatbot</h1>
    <div id="chatbox"></div>
    <input type="text" id="userInput" placeholder="Type your message...">
    <button onclick="sendMessage()">Send</button>

    <script>
        async function sendMessage() {
            const input = document.getElementById('userInput').value;
            document.getElementById('chatbox').innerHTML += `<div>User: ${input}</div>`;
            document.getElementById('userInput').value = '';

            const response = await fetch('http://127.0.0.1:5000/api/chat', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ message: input })
            });

            const data = await response.json();
            document.getElementById('chatbox').innerHTML += `<div>Bot: ${data.reply}</div>`;
        }
    </script>
</body>
</html>
"""

display(HTML(html_code))